Choosing an angle before looking at the data would be an uninformed choice, and most likely a forced one: you decide what you want, then try to make the data support it. So the first step is to explore the API and find out what it gives us access to. Once we know that, we can see what we can actually do with it.

In [ ]:
import re
import json, urllib.request, urllib.error
import duckdb
import pandas
from pydantic import BaseModel

pandas.set_option("display.max_colwidth", None)
pandas.set_option("display.max_rows", 100)

The API's help page lists every endpoint in one table. `pandas.read_html` reads every table on a web page and returns a list with one DataFrame per table. The help page has one table, so we take the first, `[0]`. In: the help page's URL. Out: a DataFrame with one row per endpoint.

In [ ]:
HelpTable: pandas.DataFrame = pandas.read_html("https://api.erg.ic.ac.uk/AirQuality/help")[0]
HelpTable

Each row is one endpoint: a Uri, with the parameters it needs written in braces, an HTTP method and a description. Most endpoints appear more than once, in different formats. A Uri with no suffix returns XML, and the others end in /Json, /csv or /pdf.

To work with one endpoint at a time, we turn the table into a list of rows. `to_dict` can lay a table out in several ways, and its argument picks which one. `"records"` means one dictionary per row, with the column names as its keys (a record is another word for a row). In: a DataFrame of 70 rows. Out: a list of 70 dictionaries, like the first one below.

In [ ]:
HelpRows: list[dict[str, str]] = HelpTable.to_dict("records")
print(len(HelpRows))
HelpRows[0:36]

Written as a type, an endpoint is three pieces of text. Pydantic checks that anything built as an `Endpoint` has these three fields, each one text, and fails straight away if one is missing or isn't text. From here on, `Endpoint` (capital E) is the type, `Endpoints` is the list of them, and `endpoint` (lower case) is one of them inside a loop.

In [ ]:
class Endpoint(BaseModel):
    Uri: str
    Method: str
    Description: str

Each row becomes an `Endpoint`. `**row` unpacks a dictionary into named arguments, so `Endpoint(**row)` is the same as `Endpoint(Uri=row["Uri"], Method=row["Method"], Description=row["Description"])`. In: 70 dictionaries. Out: 70 Endpoints, each one checked against the type.

In [ ]:
Endpoints: list[Endpoint] = [Endpoint(**row) for row in HelpRows]
print(len(Endpoints))
Endpoints[0:]

We keep only the JSON endpoints, because Python reads JSON straight into dictionaries and lists. In: 70 Endpoints. Out: the ones whose Uri ends in /Json.

In [ ]:
JsonEndpoints: list[Endpoint] = [endpoint for endpoint in Endpoints if endpoint.Uri.endswith("/Json")]
len(JsonEndpoints)

`ParametersOf` maps an endpoint to the set of parameters it needs: the names in braces in its Uri. In: one Endpoint. Out: a set of names, for example {"GroupName", "Year"}.

In [ ]:
def ParametersOf(endpoint: Endpoint) -> set[str]:
    return set(re.findall(r"\{(\w+)\}", endpoint.Uri))

`ParametersOf` applied to every JSON endpoint, as a table. In: 36 Endpoints. Out: a DataFrame of 36 rows, one Uri and its parameters per row. An endpoint whose parameters are set(), the empty set, needs nothing, so we can call it straight away.

In [ ]:
ParametersTable: pandas.DataFrame = pandas.DataFrame(
    [(endpoint.Uri, ParametersOf(endpoint)) for endpoint in JsonEndpoints], columns=["Uri", "Parameters"])
ParametersTable

Every parameter the API uses: the union of `ParametersOf` over all the JSON endpoints. These are the values we have to find before we can call every endpoint. In: 36 Endpoints. Out: one set of names.

In [ ]:
AllParameters: set[str] = set().union(*[ParametersOf(endpoint) for endpoint in JsonEndpoints])
print(len(AllParameters))
AllParameters

The chain so far, each name with its shape:

```
help page URL
→ HelpTable         DataFrame: 70 rows × (Uri, Method, Description)
→ HelpRows          list of 70 dicts, one per row
→ Endpoints         list of 70 Endpoint, each one checked
→ JsonEndpoints     list of 36 Endpoint
→ ParametersTable   DataFrame: 36 rows × (Uri, Parameters)
→ AllParameters     set of 25 names

ParametersOf: Endpoint → set of names, used to make the last two
```

Now we find out what we can call. `Known` is the set of parameters we have values for, and at the start it is empty. An endpoint can be called once every parameter it needs is known, ParametersOf(endpoint) ⊆ Known, which Python writes as `<=`. In: a set of known parameter names. Out: the endpoints we can call with them.

In [ ]:
def CallableWith(Known: set[str]) -> list[Endpoint]:
    return [endpoint for endpoint in JsonEndpoints if ParametersOf(endpoint) <= Known]
# return each endpoint in jsonendpoints if that endpoints parameters are a subset of known
Known: set[str] = set()
[endpoint.Uri for endpoint in CallableWith(Known)]

`Get` calls an endpoint. Each Uri in the help table is already a template: `format` fills its braces with the values we pass, so `/Information/MonitoringSites/GroupName={GroupName}/Json` with `GroupName="London"` becomes `/Information/MonitoringSites/GroupName=London/Json`. In: a Uri and a value for each of its parameters. Out: the reply, read from JSON into dictionaries and lists.

In [ ]:
def Get(Uri: str, **Values: str) -> dict:
    return json.load(urllib.request.urlopen("https://api.erg.ic.ac.uk/AirQuality" + Uri.format(**Values)))

Every reply wraps its records differently: Groups in `{"Groups": {"Group": [...]}}`, Species in `{"AirQualitySpecies": {"Species": [...]}}`. Hard-coding those keys for each endpoint works, but there is a pattern behind them. The API builds its JSON from XML: every reply has one root, keys that start with @ are attributes, and the other keys are child elements. So the records are the root's one child that isn't an attribute. That's a rule, and we can write it once instead of once per endpoint.

This is a decision with several options:

```
Finding the records  { hard-code each path | a table of paths ✓ relying | a convention rule, checked ✓ exploring | search for the biggest list }
```

We use the rule to explore, because it works on endpoints we haven't seen yet, and fails loudly where the convention stops. Later, for the endpoints we rely on, we write the paths down as a table and check it. Searching for the biggest list is the option to avoid: it guesses, and a wrong guess is silent.

First, one piece of mess that comes from the XML too. XML has no lists, only repeated elements, so a list with one item comes back as a single dictionary: one record looks like `{...}`, and several look like `[{...}, {...}]`. `AsList` makes both a list. In: a dictionary or a list. Out: a list.

In [ ]:
def AsList(Value: dict | list) -> list:
    return Value if isinstance(Value, list) else [Value]

`Records` is the rule. `[Root] = Reply.values()` says there is exactly one root, and raises an error if there isn't. The same goes for the one child. So the rule checks its own assumption every time, instead of guessing. In: a reply. Out: its list of records.

In [ ]:
def Records(Reply: dict) -> list[dict]:
    [Root] = Reply.values()                                                          # exactly one root
    [Children] = [Value for Key, Value in Root.items() if not Key.startswith("@")]   # exactly one child that isn't an attribute
    return AsList(Children)

We call every endpoint we can, and count the records in each reply, so an empty reply would show up as 0 instead of hiding behind "ok". The help page says none of these six needs a parameter, but two of them answer 400 Bad Request. The help page is a claim about the API, and calling each endpoint is what checks it.

In [ ]:
for endpoint in CallableWith(Known):
    try:
        print("ok ", len(Records(Get(endpoint.Uri))), "records", endpoint.Uri)
    except urllib.error.HTTPError as Error:
        print(Error.code, endpoint.Uri)

Two more pieces of mess, shared by every endpoint. Field names start with @, so the field `@SiteCode` and the parameter `SiteCode` are two names for one thing. And a missing value is an empty string, not a null. `Clean` fixes both at the boundary: it removes the @, and turns "" into a real missing value. After this, a field and its parameter have one name. In: a table from the API. Out: the same table, with clean names and real missing values.

In [ ]:
def Clean(Table: pandas.DataFrame) -> pandas.DataFrame:
    return Table.rename(columns=lambda Name: Name.removeprefix("@")).replace("", None)

The steps compose. `Get` calls the endpoint, `Records` finds its records, `json_normalize` makes them a table with one row per record, and `Clean` fixes the names and missing values. `GetTable` is that chain: GetTable = Clean ∘ json_normalize ∘ Records ∘ Get. In: a Uri and a value for each of its parameters. Out: a clean table.

In [ ]:
def GetTable(Uri: str, **Values: str) -> pandas.DataFrame:
    return Clean(pandas.json_normalize(Records(Get(Uri, **Values))))

Groups and Species are the replies we can use next.

In [ ]:
GroupsTable: pandas.DataFrame = GetTable("/Information/Groups/Json")
GroupsTable

In [ ]:
SpeciesTable: pandas.DataFrame = GetTable("/Information/Species/Json")
SpeciesTable

This is how endpoints connect. A table gives values for a parameter when one of its columns has that parameter's name: the GroupName column of `GroupsTable` holds values we can pass as GroupName. Because `Clean` removed the @, that's a plain set intersection. `Reveals` is that rule. In: a table. Out: the parameter names among its columns.

In [ ]:
def Reveals(Table: pandas.DataFrame) -> set[str]:
    return AllParameters & set(Table.columns)

print(Reveals(GroupsTable))
print(Reveals(SpeciesTable))

Known grows by what the tables reveal, and more endpoints become callable. Dates and years are different: no table reveals them, because we choose them ourselves, so we add them by hand.

In [ ]:
Known = Known | Reveals(GroupsTable) | Reveals(SpeciesTable) | {"StartDate", "EndDate", "Date", "Year"}
print(Known)
[endpoint.Uri for endpoint in CallableWith(Known)]

The discovery so far:

```
Known = set()
→ CallableWith(Known)   6 endpoints, 4 of which answer
→ GetTable              GroupsTable (69 rows), SpeciesTable (6 rows)
→ Reveals               {GroupName}, {SpeciesCode, SpeciesName}
→ Known grows           + the dates we choose
→ CallableWith(Known)   16 endpoints, MonitoringSites among them

GetTable = Clean ∘ json_normalize ∘ Records ∘ Get
```

Second round. MonitoringSites is now callable, and it lists the sites in a group. `GroupsTable` describes London as "All currently active boroughs of London", so that's the group we use. In: a GroupName. Out: a table with one row per site.

In [ ]:
SitesTable: pandas.DataFrame = GetTable("/Information/MonitoringSites/GroupName={GroupName}/Json", GroupName="London")
print(Reveals(SitesTable))
SitesTable

`SitesTable` reveals SiteCode, so Known grows again.

In [ ]:
Known = Known | Reveals(SitesTable)
print(len(CallableWith(Known)))
[endpoint.Uri for endpoint in CallableWith(Known)]

Data/SiteSpecies, the hourly readings, is now callable, because we know SiteCode and SpeciesCode. But knowing the names isn't knowing which values go together. Any site code with any species code makes a request that looks valid, and the API answers 200 even when the pair makes no sense, with every value empty (see Mess.md). MonitoringSiteSpecies says which pairs are real: for each site, the species it measures and the dates it started and stopped.

The pairs sit one level below the records: each site has a Species field, which `AsList` makes a list. Then `json_normalize` can make one row per pair: `record_path` names the list to turn into rows, and `meta` the site fields to copy onto each row. Which level we want is a choice about meaning, so this path is written by hand. In: the reply. Out: a table with one row per (site, species) pair.

In [ ]:
SiteSpeciesSites: list[dict] = Records(Get("/Information/MonitoringSiteSpecies/GroupName={GroupName}/Json", GroupName="London"))
for site in SiteSpeciesSites:
    site["Species"] = AsList(site["Species"])

SiteSpeciesTable: pandas.DataFrame = Clean(pandas.json_normalize(SiteSpeciesSites, record_path="Species", meta=["@SiteCode"]))
SiteSpeciesTable

The valid pairs as a set: a relation between sites and species, ValidPairs ⊆ Sites × Species. `zip` pairs the two columns row by row. The table has 650 rows but fewer distinct pairs, because HS1 and HS5 are listed more than once. Most possible pairs aren't in the set, and each of those is a request that would come back empty without an error. In: `SiteSpeciesTable`. Out: a set of (SiteCode, SpeciesCode) pairs.

In [ ]:
ValidPairs: set[tuple[str, str]] = set(zip(SiteSpeciesTable["SiteCode"], SiteSpeciesTable["SpeciesCode"]))
print(len(ValidPairs), "valid pairs out of", len(SitesTable) * len(SpeciesTable), "possible")

Now a decision: where do we stop a bad request? A request passes through several boundaries, and we could check at any of them. Each line below is one boundary, the braces hold its options, and ✓ marks what we chose.

```
Parameters     { nothing | format check | site ∈ SitesTable | (site, species) ∈ ValidPairs ✓ | dates within Started–Finished }
      ↓
The call       { catch everything | catch HTTPError and log it ✓ exploring | let it raise ✓ readings | retry }
      ↓
Reply content  { trust it | no values at all → raise ✓ | a threshold on % missing | a Pydantic model of the reply }
      ↓
Storage        { column types ✓ | primary key ✓ | CHECK constraints }                                   pipeline.py
      ↓
Across runs    { upsert and a changes table ✓ | a runs log ✓ | last measured hour per site ✓ }          pipeline.py
```

We need two checks, because each one catches what the other can't. Membership in `ValidPairs` stops a pair the API doesn't list, before we call. But listed doesn't mean it has data: MY1 stopped measuring PM10 in 2019, and has had no CO since 30 Oct, though CO is still listed with no end date. Only looking at the reply catches those.

What we learned, written down. The rule found the records in every reply we used. For the endpoints the pipeline relies on, we write those paths down as a table: a belief about the API's shape, kept in one place, which code can follow without guessing. In: nothing. Out: a dictionary from each Uri to the keys that lead to its records.

In [ ]:
ReadingsUri: str = "/Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json"

RecordPaths: dict[str, list[str]] = {
    "/Information/MonitoringSites/GroupName={GroupName}/Json":       ["Sites", "Site"],
    "/Information/MonitoringSiteSpecies/GroupName={GroupName}/Json": ["Sites", "Site"],
    ReadingsUri:                                                     ["RawAQData", "Data"],
}

`Follow` walks a path down a reply, one key at a time. In: a reply and a path. Out: the records at the end of the path.

In [ ]:
def Follow(Reply: dict, Path: list[str]) -> list[dict]:
    Value = Reply
    for Key in Path:
        Value = Value[Key]
    return AsList(Value)

A table of beliefs can go stale without telling us, so we check it against the API: call each endpoint with sample values, and make sure its path leads to the same records the rule finds. A key that has gone missing fails on its own, with a KeyError. `format` ignores values a Uri doesn't use, so one set of samples serves every endpoint. In: `RecordPaths`. Out: one line per endpoint, or an error naming the belief that no longer holds.

In [ ]:
Samples: dict[str, str] = {"GroupName": "London", "SiteCode": "MY1", "SpeciesCode": "NO2", "StartDate": "2026-10-31", "EndDate": "2026-11-01"}

for Uri, Path in RecordPaths.items():
    Reply = Get(Uri, **Samples)
    if Follow(Reply, Path) != Records(Reply):
        raise ValueError(f"{Path} no longer leads to the records of {Uri}")
    print("holds", len(Follow(Reply, Path)), "records", Uri)

`GetReadings` follows the stated path, with both checks around the call. In: a SiteCode, SpeciesCode, StartDate and EndDate. Out: a clean table with one row per hour, or an error saying which check failed.

In [ ]:
def GetReadings(**Values: str) -> pandas.DataFrame:
    # before the call: is the pair one the API lists?
    if (Values["SiteCode"], Values["SpeciesCode"]) not in ValidPairs:
        raise ValueError(f"Not a listed pair: {Values['SiteCode']} × {Values['SpeciesCode']}")
    Rows = Follow(Get(ReadingsUri, **Values), RecordPaths[ReadingsUri])
    # after the call: listed doesn't mean it has data
    if not any(row["@Value"] != "" for row in Rows):
        raise ValueError(f"No values: {Values['SiteCode']} × {Values['SpeciesCode']}, {Values['StartDate']} to {Values['EndDate']}")
    return Clean(pandas.json_normalize(Rows)).assign(**Values)

Five requests for the same day, one for each kind of outcome: a real pair with data, a site that doesn't exist, a species the site doesn't measure, a species it stopped measuring in 2019, and one it still lists but stopped on 30 Oct (true on 1 Nov 2026).

In [ ]:
for site, species in [("MY1", "NO2"), ("ZZZ", "NO2"), ("GT1", "CO"), ("MY1", "PM10"), ("MY1", "CO")]:
    try:
        print("ok", site, species, len(GetReadings(SiteCode=site, SpeciesCode=species, StartDate="2026-10-31", EndDate="2026-11-01")), "hours")
    except ValueError as Error:
        print(Error)

Where each piece of mess lives. Variation that differs from endpoint to endpoint sits in a table. Variation with a pattern sits in a rule, which is a table compressed into one line. Either way it has one home, at the boundary, and the code after it is the same for every endpoint. Everything we found is in Mess.md.

```
Mess                                                    Where it lives
each endpoint wraps its records under different keys    Records, a rule
the paths of the endpoints we rely on                   RecordPaths, a table, checked against the API
a list with one item comes back as a dictionary         AsList, a rule
field names start with @, parameters don't              Clean, a rule
a missing value is ""                                   Clean, a rule
any site × species looks valid, most have no data       ValidPairs, a relation, checked before the call
a listed pair can still have no data                    the reply check, after the call
```

The second round:

```
Known + Reveals(SitesTable)   26 callable endpoints, Data/SiteSpecies among them
→ Records, AsList, Clean      SiteSpeciesTable: 650 rows, one per (site, species)
→ zip, set                    ValidPairs: 646 pairs out of 1,548 possible
→ RecordPaths, Follow         the paths we rely on, checked against the API
→ GetReadings                 check the pair → call → check the reply → a clean table of hours
```

Collecting history. `pipeline.py` is the last steps of this notebook, made to run again and again: the same `Get`, `AsList`, `Follow` with `RecordPaths`, `Clean`, and `GetReadings` with both checks, then a save to air.duckdb. Each run fetches the last 7 days of hourly NO₂ for MY1 and GT1. Run `python pipeline.py` first, because this notebook only reads the database. The 7 days overlap from one run to the next, and the table's primary key means each hour is stored once: a new hour is inserted, and an hour we already have is updated. Each run is logged in `runs`.

In [ ]:
Database = duckdb.connect("air.duckdb", read_only=True)
Database.sql("SELECT * FROM runs ORDER BY RunAt")

`changes` records every stored value that a later run changed. So far each one is a late arrival: an hour that was empty when it was first fetched, and filled in by a later run. None is yet a correction of a real value.

In [ ]:
Database.sql("SELECT * FROM changes ORDER BY RunAt")

Shaping. The API sends everything as text, with @ in front of field names and "" for a missing value. `Clean` fixes the names and the missing values, and the `readings` table gives each column a type, a timestamp and a number. Its primary key says what one row is: one site, one species, one hour, and its value. The request window isn't part of a reading, so it lives in `runs`. `sites` is the London site list, refreshed on every run, and joins to `readings` on SiteCode, which has the same name in both.

In [ ]:
Database.sql("DESCRIBE readings")

Using it. The question: how much worse is the air right beside a busy road than nearby, and at what time of day? `sites` says MY1, on Marylebone Road, is Kerbside, and GT1, Station Walk, is Urban Background. `HourlyTable` puts them side by side, one row per hour, and `BothTable` keeps only the hours where both sites have a value, so a gap at one site can't skew the comparison. In: `readings` joined to `sites`. Out: a table with one row per hour and a column per site type.

In [ ]:
HourlyTable: pandas.DataFrame = Database.sql("""
SELECT MeasurementDateGMT AS Hour,
       avg(Value) FILTER (WHERE SiteType = 'Kerbside')         AS Kerbside,
       avg(Value) FILTER (WHERE SiteType = 'Urban Background') AS Background
FROM readings JOIN sites USING (SiteCode)
WHERE SpeciesCode = 'NO2'
GROUP BY Hour
ORDER BY Hour
""").df()
BothTable: pandas.DataFrame = HourlyTable.dropna()
print(len(BothTable), "of", len(HourlyTable), "hours have a value at both sites")

Over those hours, the kerbside mean against the background mean. DuckDB can query a DataFrame by its name, so `BothTable` works as a table in SQL.

In [ ]:
Database.sql("""
SELECT count(*) AS Hours, round(avg(Kerbside), 1) AS Kerbside, round(avg(Background), 1) AS Background,
       round(avg(Kerbside) / avg(Background), 1) AS Ratio
FROM BothTable
""")

The same by hour of day. Gap is the kerbside mean minus the background mean.

In [ ]:
ByHourOfDay: pandas.DataFrame = Database.sql("""
SELECT hour(Hour) AS HourOfDay, round(avg(Kerbside), 1) AS Kerbside, round(avg(Background), 1) AS Background,
       round(avg(Kerbside) - avg(Background), 1) AS Gap
FROM BothTable
GROUP BY HourOfDay
ORDER BY HourOfDay
""").df().set_index("HourOfDay")
ByHourOfDay

As a chart. This also saves no2_by_hour.png for the README.

In [ ]:
Chart = ByHourOfDay[["Kerbside", "Background"]].plot(marker="o", title="Mean NO₂ by hour of day: Marylebone Road (kerbside) and Station Walk (background)",
                                                     xlabel="Hour of day (GMT)", ylabel="Mean NO₂ (µg/m³)")
Chart.figure.savefig("no2_by_hour.png", bbox_inches="tight")

Close the database, so that `pipeline.py` can write to it again.

In [ ]:
Database.close()